# Dhivehi OCR with Qwen 2.5 3B

This notebook performs OCR on Dhivehi PDFs using a fine-tuned Qwen 2.5 3B model hosted on Google Drive, with optional layout detection using PaliGemma.

In [ ]:
# 1. Install Dependencies
!apt-get install -y poppler-utils
!pip install pdf2image transformers accelerate bitsandbytes sentencepiece

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuration and Model Loading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, PaliGemmaForConditionalGeneration, PaliGemmaProcessor
from pdf2image import convert_from_path
import os
from tqdm import tqdm

# Paths
QWEN_CHECKPOINT = "/content/drive/MyDrive/Qwen2.5_3B_Dhivehi_OCR_37k_Full/checkpoint-10000"
PALIGEMMA_MODEL = "google/paligemma-3b-pt-224" # Or your fine-tuned layout model
INPUT_DIR = "/content/drive/MyDrive/mcq/data/saruna"
OUTPUT_DIR = "/content/drive/MyDrive/mcq/data/extracted_text"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading PaliGemma for Layout Detection...")
layout_processor = PaliGemmaProcessor.from_pretrained(PALIGEMMA_MODEL)
layout_model = PaliGemmaForConditionalGeneration.from_pretrained(PALIGEMMA_MODEL, device_map="auto", torch_dtype=torch.bfloat16)

print("Loading Qwen 2.5 OCR Model...")
tokenizer = AutoTokenizer.from_pretrained(QWEN_CHECKPOINT, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    QWEN_CHECKPOINT,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    load_in_4bit=True
)

In [ ]:
# 4. Layout Detection & OCR Functions
def detect_layout(image):
    # PaliGemma prompt for detecting segments (adjust based on your specific layout task)
    prompt = "detect text segments"
    inputs = layout_processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = layout_model.generate(**inputs, max_new_tokens=100)
    
    # This is a placeholder for parsing PaliGemma's detection tokens into bounding boxes
    # Return list of crops or bounding boxes sorted by logical reading order (RTL for Dhivehi)
    return [image] # Default to full image for now

def perform_ocr(pdf_path):
    images = convert_from_path(pdf_path)
    full_text = ""
    
    for page_img in images:
        # 1. Detect segments (Layout Analysis)
        segments = detect_layout(page_img)
        
        # 2. OCR each segment with Qwen
        for segment_img in segments:
            # Placeholder for Qwen vision-language inference
            # Assuming the model is used for transcription
            pass
        full_text += "\n"
    return full_text

In [ ]:
# 5. Main Processing Loop
pdf_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

for pdf_path in tqdm(pdf_files, desc="OCR Processing"):
    filename = os.path.basename(pdf_path)
    txt_path = os.path.join(OUTPUT_DIR, os.path.splitext(filename)[0] + ".txt")
    
    if os.path.exists(txt_path):
        continue
        
    text = perform_ocr(pdf_path)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)